# Approximate Posterior Model — Prior-Driven Drift Fitting

Fits the approximate posterior memory model (Hypothesis 2: Prior-Based Account).
Memory traces drift toward prior modes via the score function ∇ log π(x).

**Key difference from the three-regime model:** No noise regimes. A single
noise parameter σ governs all memory noise, plus `drift_step_size` controls
the strength of prior-driven drift.

| Stage | Parameter | Method | ISIs |
|-------|-----------|--------|------|
| A | `sigma0` (encoding noise) | ISI-0 toy experiments | [0] |
| B | `sigma` (drift noise) | Compact multi-ISI sequences | [1, 2, 4, 8, 16, 32, 64] |

The `drift_step_size` can be searched jointly or in a third stage.

In [ ]:
import sys, os, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict
from scipy.spatial.distance import pdist

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.plotting import ensure_dir
from utls.loading import (
    load_results_with_exclusion_2,
    move_sequences_to_used,
    load_results_with_exclusion_no_dropping,
)
from utls.runners_v2 import (
    run_experiment_scores_prior,
    make_noise_schedule,
)
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.io_utils import (
    make_model_save_dir,
    save_all_figures,
    save_single_figure,
    save_runs_summary,
)
from encoders import *

from utls.toy_experiments import (
    make_toy_experiment_list,
    make_compact_multi_isi_sequences,
    infer_trial_isis,
    make_high_diversity_sequences,
)
from utls.sigma_fitting import (
    log_mid,
    fit_sigma_1d,
    plot_sigma_fit,
)

from ScoreFunction import ScoreFunction

## 1. Load config & data

In [ ]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path


# --- Use a texture-based config ---
CONFIG_PATH = (
    "/om2/user/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)

model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)

In [ ]:
# ---- experiment ----
exp_cfg = model_cfg["experiment"]
which_task = exp_cfg["which_task"]
is_multi = exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)

isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model: constant (single regime) for this model ----
noise_mode = "constant"

# ---- representation: texture PCA 256D ----
repr_cfg = model_cfg["representation"]
time_avg = repr_cfg["time_avg"]
encoder_type = repr_cfg.get("type", "texture")
layer = repr_cfg.get("layer", None)
pc_dims = repr_cfg.get("pc_dims", 256)

# ---- load human data ----
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)

human_curve = compute_human_curve(human_runs, is_multi, which_isi)
print("ISIs:", isis)
print("Human d' curve:", human_curve)
print(f"Total real sequences: {len(exp_list)}")

## 2. Build encoder & encode stimuli

In [ ]:
NN_ENCODERS = {"kell2018", "resnet50"}
encoder_task = (
    "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
)

encoder_cfg = dict(
    encoder_type=encoder_type,
    model_name=encoder_type,
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    time_avg=time_avg,
    device="cuda",
)

if encoder_type in NN_ENCODERS:
    encoder_cfg["layer"] = layer
if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)
X_np = X.detach().cpu().numpy()
print("Encoded shape:", X_np.shape, "  any NaN?", torch.isnan(X).any().item())

## 3. Load score function (prior)

In [ ]:
SCORE_CONFIG = "/om2/user/bjmedina/auditory-memory/memory/assets/bryan.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

score_fn = ScoreFunction(
    mode="textures",
    restart=True,
    config=SCORE_CONFIG,
    device=DEVICE,
)

# Quick sanity check: forward pass on a random input
test_input = torch.randn(1, 256, device=DEVICE)
test_output = score_fn.forward(test_input)
print(f"Score output shape: {test_output.shape}")
print(f"Score output norm: {test_output.view(-1).norm():.4f} (should be ~1.0)")

## 4. Parameter bounds & stimulus pool

In [ ]:
param_bounds = {
    "sigma0": (0.0001, 22),
    "sigma": (0.0001, 10),
    "drift_step_size": (0.0001, 1.0),
}

print("Parameter bounds:")
for k, v in param_bounds.items():
    print(f"  {k}: ({v[0]:.6f}, {v[1]:.6f})")

# Stimulus pool for sequence generation
stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f"\nStimulus pool size: {len(stimulus_pool)}")
assert len(stimulus_pool) >= 65, (
    f"Stimulus pool ({len(stimulus_pool)}) too small for ISI-64 blocks (need >= 65)"
)

## 5. Human d' targets

In [ ]:
isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}

sigma0_human = {0: float(human_curve[isi_to_hc_idx[0]])}
sigma_human = {
    isi: float(human_curve[isi_to_hc_idx[isi]])
    for isi in [1, 2, 4, 8, 16, 32, 64]
}

print("Stage A targets (sigma0 — encoding noise):")
for isi, dp in sigma0_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

print("\nStage B targets (sigma — drift noise):")
for isi, dp in sigma_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

# Initial values
sigma_init = log_mid(*param_bounds["sigma"])
drift_init = log_mid(*param_bounds["drift_step_size"])
print(f"\nInitial sigma = {sigma_init:.6f}")
print(f"Initial drift_step_size = {drift_init:.6f}")

---
## Stage A: Fit sigma0 (ISI = 0, toy experiments)

ISI=0 means immediate repeats — only one step of noise, so only sigma0
matters. Standard toy experiments (pairs of identical stimuli) are
efficient here. Note: drift has minimal effect at ISI=0.

In [ ]:
isi0_exps = {
    0: make_toy_experiment_list(
        stimulus_pool, isi=0, n_experiments=100, k_stimuli=5, seed=0
    )
}
print(f"ISI-0 toy experiments: {len(isi0_exps[0])} exps, "
      f"avg len {np.mean([len(e) for e in isi0_exps[0]]):.0f} trials")

N_REFINE_ITERS = 2
N_MC = 8

# For stage A, drift is negligible, but we pass it for API consistency
stage_a = fit_sigma_1d(
    run_experiment_fn=run_experiment_scores_prior,
    sigma_name="sigma0",
    sigma_bounds=param_bounds["sigma0"],
    fixed_sigmas={"sigma0": sigma_init},  # sigma0 = drift noise, overridden by grid
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiments_by_isi=isi0_exps,
    human_dprimes_by_isi=sigma0_human,
    t_step=None,
    n_grid=15,
    n_mc=N_MC,
    n_refine_iters=N_REFINE_ITERS,
    seed=0,
)

sigma0_fitted = stage_a["best_sigma"]
print(f"\n>>> sigma0 = {sigma0_fitted:.6f}  (MSE = {stage_a['best_mse']:.6f})")
plot_sigma_fit(stage_a, human_dprimes_by_isi=sigma0_human,
               title="Stage A: sigma0 (ISI = 0)");

---
## Stage B: Fit sigma (drift noise) with fixed drift_step_size

Using compact multi-ISI sequences covering ISIs [1, 2, 4, 8, 16, 32, 64].
We fix drift_step_size to an initial value and fit the noise magnitude.

In [ ]:
SIGMA_ISIS = [1, 2, 4, 8, 16, 32, 64]
SIGMA_LENGTH = 75
SIGMA_N_SEQS = 20
SIGMA_MIN_PAIRS = 5

sigma_exps, sigma_isi_keys = make_high_diversity_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=SIGMA_ISIS,
    n_sequences=SIGMA_N_SEQS,
    length=SIGMA_LENGTH,
    min_pairs_per_isi=SIGMA_MIN_PAIRS,
    seed=1000,
)

print(f"Generated {len(sigma_exps)} compact sequences")
print(f"Trials per sequence: {len(sigma_exps[0])}")

# Validate ISI distribution
sigma_isi_counts = defaultdict(list)
for seq in sigma_exps:
    counts = Counter(infer_trial_isis(seq))
    for isi_val in SIGMA_ISIS:
        sigma_isi_counts[isi_val].append(counts.get(isi_val, 0))

print("\nPairs per ISI per sequence (mean +/- std):")
for isi_val in SIGMA_ISIS:
    vals = sigma_isi_counts[isi_val]
    print(f"  ISI {isi_val:>2}: {np.mean(vals):.1f} +/- {np.std(vals):.1f}  "
          f"(min={min(vals)}, max={max(vals)})")

In [ ]:
# NOTE: fit_sigma_1d passes extra kwargs through to run_experiment_fn.
# We need to pass score_model and drift_step_size.
# This requires that fit_sigma_1d forwards unknown kwargs,
# or we wrap run_experiment_scores_prior with functools.partial.

from functools import partial

DRIFT_STEP_SIZE = drift_init  # initial drift; can be tuned later

run_fn_with_prior = partial(
    run_experiment_scores_prior,
    score_model=score_fn,
    drift_step_size=DRIFT_STEP_SIZE,
)

stage_b = fit_sigma_1d(
    run_experiment_fn=run_fn_with_prior,
    sigma_name="sigma0",  # In constant noise mode, sigma0 IS the drift noise
    sigma_bounds=param_bounds["sigma"],
    fixed_sigmas={},  # no other sigmas to fix
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    human_dprimes_by_isi=sigma_human,
    t_step=None,
    n_grid=10,
    n_mc=N_MC,
    n_refine_iters=N_REFINE_ITERS,
    seed=100_000,
    experiment_list=sigma_exps,
    isi_keys=sigma_isi_keys,
    target_isis=SIGMA_ISIS,
    n_seqs_per_rep=SIGMA_N_SEQS // 2,
)

sigma_fitted = stage_b["best_sigma"]
print(f"\n>>> sigma = {sigma_fitted:.6f}  (MSE = {stage_b['best_mse']:.6f})")
plot_sigma_fit(stage_b, human_dprimes_by_isi=sigma_human,
               title="Stage B: sigma (drift noise, all ISIs)");

---
## 6. Fitted parameter summary

In [ ]:
print("=" * 50)
print("APPROXIMATE POSTERIOR MODEL FIT")
print("=" * 50)
print(f"  sigma0 (encoding)   = {sigma0_fitted:.6f}  (Stage A MSE: {stage_a['best_mse']:.6f})")
print(f"  sigma  (drift noise) = {sigma_fitted:.6f}  (Stage B MSE: {stage_b['best_mse']:.6f})")
print(f"  drift_step_size      = {DRIFT_STEP_SIZE:.6f}")

---
## 7. Final evaluation on ALL real sequences

Evaluate the fitted parameters on every real participant sequence.

In [ ]:
%%time

# Run the fitted model on the real experiment sequences
final_run = run_experiment_scores_prior(
    sigma0=sigma0_fitted,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiment_list=exp_list,
    score_model=score_fn,
    drift_step_size=DRIFT_STEP_SIZE,
    seed=42,
)

model_dp = compute_model_dprime_for_run(final_run, isis)

print("Model d' curve:", model_dp)
print("Human d' curve:", human_curve)

## 8. Summary plot: model vs human d' across all ISIs

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

isi_labels = [str(i) for i in isis]
x_pos = np.arange(len(isis))

ax.plot(x_pos, human_curve, 'ko-', label='Human', linewidth=2, markersize=8)
ax.plot(x_pos, model_dp, 's--', color='tab:blue',
        label=f'Approx. Posterior (σ={sigma_fitted:.3f}, drift={DRIFT_STEP_SIZE:.4f})',
        linewidth=2, markersize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels(isi_labels)
ax.set_xlabel('ISI (intervening items)')
ax.set_ylabel("d'")
ax.set_title('Approximate Posterior Model vs Human Performance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()